# MCH–MFT Track Matching — Training & Testing

Learn-to-Rank (XGBoost LambdaRank) to score (MCH, MFT candidate) pairs.  
Each MCH track forms a **group** of up to 5 candidates; the model learns to rank the true match highest within each group.
The top-ranked candidate is accepted if its score exceeds a tunable threshold.

**Stages:**
1. Load data with hipe4ml
2. Feature engineering
3. Train/test split (by MCH track group)
4. XGBoost LambdaRank training
5. Group-level evaluation (efficiency, purity, MRR)

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils

from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 1. Load, Format, Engineer Data

In [ ]:
df_original = Utils.get_dataframe("FwdMatchMLCandidatesFull.root")


In [ ]:
df =df_original.copy() 
importlib.reload(Utils)

In [ ]:
df = Utils.process_dataframe(df, makedummies=True)

FEATURES = [f for f in df.columns.tolist() if f not in Utils.NON_TRAINING_FEATURES]

TARGET = "IsSignal"

GROUP  = "mchID"

In [ ]:
df[FEATURES].describe()

In [ ]:
FEATURES

## 2. Sanity Checks

In [ ]:
n_mch_tracks = df["mchID"].nunique()
n_positive = df["IsSignal"].sum()
candidates_per_track = df.groupby("mchID").size()

print(f"MCH tracks:          {n_mch_tracks:,}")
print(f"Total pairs:         {len(df):,}")
print(f"True matches:        {int(n_positive):,} ({100*n_positive/len(df):.1f}%)")
print(f"Candidates per track: min={candidates_per_track.min()}, "
      f"max={candidates_per_track.max()}, "
      f"mean={candidates_per_track.mean():.2f}")

# Tracks with no true match among candidates
tracks_with_match = df.groupby("mchID")["IsSignal"].max()
n_no_match = (tracks_with_match == 0).sum()
print(f"Tracks with no true match in candidates: {n_no_match:,} "
      f"({100*n_no_match/n_mch_tracks:.1f}%)")

## 4. Train / Test Split

Split is done **by MCH track group**, not by row, to avoid data leakage  
(candidates from the same MCH track must not appear in both train and test).

In [ ]:

groups = df[GROUP].values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=groups))

df_train = df.iloc[train_idx].copy()
df_test  = df.iloc[test_idx].copy()

# Sort by group — required by XGBRanker
df_train = df_train.sort_values(GROUP).reset_index(drop=True)
df_test  = df_test.sort_values(GROUP).reset_index(drop=True)

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]

# Group size arrays: number of candidates per MCH track, in order
# XGBRanker needs to know which rows belong to the same query
train_groups = df_train.groupby(GROUP, sort=False).size().values
test_groups  = df_test.groupby(GROUP, sort=False).size().values

print(f"Train: {len(df_train):,} pairs ({df_train[GROUP].nunique():,} MCH tracks)")
print(f"Test:  {len(df_test):,} pairs ({df_test[GROUP].nunique():,} MCH tracks)")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")

## 5. Train XGBoost LambdaRank

`objective='rank:ndcg'` optimises the ranking within each group of candidates directly,  
avoiding the class-imbalance problem of a binary classifier and matching the problem structure exactly.  

In [ ]:
model = xgb.XGBRanker(
    objective="rank:ndcg",
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="ndcg",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_group=[train_groups, test_groups],
    verbose=50,
)

print(f"\nBest iteration: {model.best_iteration}")

# ── Training curve ────────────────────────────────────────────────────────────
results     = model.evals_result()
train_ndcg  = results["validation_0"]["ndcg"]
test_ndcg   = results["validation_1"]["ndcg"]
iterations  = range(1, len(train_ndcg) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — full training curve
ax = axes[0]
ax.plot(iterations, train_ndcg, lw=2, color="steelblue", label="Train NDCG")
ax.plot(iterations, test_ndcg,  lw=2, color="tomato",    label="Test NDCG")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("NDCG")
ax.set_title("Training curve — full")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

# Right — zoomed to the region of interest (last 20% of iterations)
zoom_start  = int(0.8 * len(train_ndcg))
ax = axes[1]
ax.plot(list(iterations)[zoom_start:], train_ndcg[zoom_start:],
        lw=2, color="steelblue", label="Train NDCG")
ax.plot(list(iterations)[zoom_start:], test_ndcg[zoom_start:],
        lw=2, color="tomato",    label="Test NDCG")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("NDCG")
ax.set_title("Training curve — zoomed (last 20%)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Gap diagnostic ────────────────────────────────────────────────────────────
final_gap = abs(train_ndcg[-1] - test_ndcg[-1])
best_gap  = abs(train_ndcg[model.best_iteration - 1] - 
                test_ndcg[model.best_iteration - 1])
print(f"Train/test NDCG gap at best iteration: {best_gap:.6f}")
print(f"Train/test NDCG gap at final iteration: {final_gap:.6f}")
print(f"{'⚠ Possible overfitting' if final_gap > 0.01 else '✓ No significant overfitting detected'}")

## 6. Feature Importance

In [ ]:
importances = pd.Series(
    model.feature_importances_, index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
importances.plot.barh(ax=ax)
ax.set_xlabel("Feature importance (gain)")
ax.set_title("XGBoost feature importances")
plt.tight_layout()
plt.show()

## 7. Group-Level Evaluation

In [ ]:
df_test = df_test.copy()
df_test["score"] = model.predict(X_test)
df_test["score_softmax"] = (
    df_test.groupby(GROUP)["score"]
    .transform(lambda s: softmax(s))
)
# df_test["score_z"] = ((df_test["score"] - df_test["score"].mean()) / df_test["score"].std())
true_match_scores = df_test.loc[df_test["MatchLabel"].isin(Utils.MATCH_LABEL_GROUPS["True match"]), "score"]

score_mean = true_match_scores.mean()
score_std  = true_match_scores.std()

df_test["score_z"] = (df_test["score"] - score_mean) / score_std

print(f"True match score  —  mean: {score_mean:.4f},  std: {score_std:.4f}")

METRICS = ["score", "score_softmax", "score_z"]

In [ ]:
g = xgb.to_graphviz(
    model,
    num_trees=10,
    graph_attr={"dpi": "300", "size": "20,10"}
)

g.render("xgb_tree", format="png", cleanup=True)

## 8. All matches

In [ ]:
match_groups = Utils.build_match_groups(df_test)
Utils.draw_all_features(features=METRICS, match_groups=match_groups, density=True)

## 9. Leading Match Metric Distribution Analysis

In [ ]:
importlib.reload(Utils)
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=METRICS, match_groups=match_groups_leader, density=True, per=0.01)

## 10. Metric score sweep

In [ ]:
def sweep_threshold_plot(
    df_eval: pd.DataFrame,
    metrics_fn,
    score_col: str = "score",
    n_steps: int = 100,
    title: str = "Metrics vs Threshold",
    Nsigma: float = 1.0,
):
    """
    Sweep threshold and plot metrics using structured DataFrame output.
    """

    score_min = df_eval[score_col].min()
    score_max = df_eval[score_col].max()
    thresholds = np.linspace(score_min, score_max, n_steps)

    all_results = []

    for t in thresholds:
        df_metrics = metrics_fn(
            df_eval,
            threshold=t,
            metric=score_col,
            Nsigma=Nsigma,
        )

        df_metrics["threshold"] = t
        all_results.append(df_metrics)
    print(f"Computed metrics are {all_results} thresholds.")

    result_df = pd.concat(all_results, ignore_index=True)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(9, 5))

    for metric, subdf in result_df.groupby("metric"):

        if subdf["value"].isna().all():
            print(f"[WARN] {metric} is all NaN → skipped")
            continue

        # Sort for clean lines
        subdf = subdf.sort_values("threshold")

        ax.plot(
            subdf["threshold"],
            subdf["value"],
            lw=2,
            label=metric,
        )

        # --- Optional uncertainty band ---
        if "uncertainty" in subdf.columns:
            ax.fill_between(
                subdf["threshold"],
                subdf["value"] - subdf["uncertainty"],
                subdf["value"] + subdf["uncertainty"],
                alpha=0.2,
            )

    ax.set_xlabel("Score threshold")
    ax.set_ylabel("Metric value")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return result_df



for entry in METRICS:
    sweep_threshold_plot(df_eval= df_test, metrics_fn = Utils.inhousemetrics, title=entry +" vs Score Threshold", score_col=entry, Nsigma=3.0)

## 11. Match Assigned Analysis

In [ ]:
# Apply threshold to get final matches, then plot feature distributions for the accepted candidates - see firsthand the contamination
threshold = 0.6
df_leader = df_test.loc[df_test.groupby("mchID")["score_softmax"].idxmax()].reset_index(drop=True)
df_leader = df_leader[df_leader["score_softmax"] >= threshold].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=FEATURES, match_groups=match_groups_leader, density=True)

## 12. Featurewise Metric breakdown

In [ ]:
importlib.reload(Utils)
for entry in Utils.DESIGNED_FEATURES:   
    Utils.plot_metrics_vs_feature(df=df_test,feature=entry, threshold = 0.4, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score_softmax", bins=25, trim_low=0.02, trim_high=0.02, Nsigma=2.0)

In [ ]:
df_debug = Utils.plot_metrics_vs_feature(df=df_test,feature='PtMCH', threshold = 0.5, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score_softmax", bins=25, trim_low=0.02, trim_high=0.02, Nsigma=1.0)

## DEBUGGING